# 01 - Exploratory Data Analysis (ApacheJIT)

**Goal:** get to know the ApacheJIT data before any modeling: its size, its columns, the label balance, the data quality, and how the labels change over time.

**Data:** `apachejit_total.csv` from the ApacheJIT replication package (Keshavarz & Nagappan, MSR 2022). It has 106,674 commits from open-source Apache Java projects, each labeled as bug-inducing or clean.

**Questions this notebook answers**
1. How big is the data, and what do the columns mean?
2. How imbalanced are the labels, and does this differ across projects?
3. Is the data clean (missing values, duplicates)?
4. Is the share of bug-inducing commits stable over time? (This decides how we split train and test.)

## 0. Setup

Import the libraries and check what is inside the raw data folder. The downloaded package is expected in a folder named `apachejit_dataset_replication`.

In [5]:
from pathlib import Path
import pandas as pd

print(list(Path("../data/raw").glob("*")))

[WindowsPath('../data/raw/.gitkeep'), WindowsPath('../data/raw/apachejit_dataset_replication')]


## 1. Load the data

**Goal:** load the full dataset and check that it is complete: size, column names and the time range.

The file we use is `apachejit_total.csv` (all commits). The train and test files that come with the package are not used here, because we build our own time-based split later.

In [6]:
df = pd.read_csv("../data/raw/apachejit_dataset_replication/apachejit/dataset/apachejit_total.csv")

print(df.shape)
print(df.columns.tolist())
print(df["year"].min(), df["year"].max())
df.head()

(106674, 18)
['commit_id', 'project', 'buggy', 'fix', 'year', 'author_date', 'la', 'ld', 'nf', 'nd', 'ns', 'ent', 'ndev', 'age', 'nuc', 'aexp', 'arexp', 'asexp']
2003 2019


,commit_id,project,buggy,fix,year,author_date,la,ld,nf,nd,ns,ent,ndev,age,nuc,aexp,arexp,asexp
0,7b8480744ea6e6fb41efd4329bb470c8f3c763db,apache/groovy,False,False,2003,1070355653,372,23,8,3,3,2.669743,0.250000,3.625000,2.125,243,243.0,0.683585
1,192b631e7be302ecde822546ba70a9853ddbda01,apache/groovy,False,False,2003,1063298262,2,2,2,2,1,1.000000,1.000000,0.000000,2.500,19,19.0,14.000000
2,0ab6465a7dece117c61c3efd3ec95d20524bdad6,apache/groovy,False,False,2003,1069704572,41,26,3,3,2,1.237612,0.666667,5.333333,45.000,233,233.0,0.000606
3,449241d5fa1aeadd5eb9fa1280d4104dc0dfb891,apache/groovy,False,False,2003,1068487625,8,6,2,1,1,0.591673,1.000000,3.000000,35.500,64,64.0,55.000000
4,698df9f5b0816e05bbf97a390428f8ef94e0cdbf,apache/groovy,False,False,2003,1063313516,70,4,6,3,1,2.519672,0.333333,0.000000,6.000,27,27.0,22.000000


**Observation:** the table has 106,674 rows and 18 columns, and covers the years 2003 to 2019. Both match the paper, so the file is complete.

| Column | Role | Meaning |
|---|---|---|
| `commit_id` | identifier | commit hash; not a feature |
| `project` | group | source repository |
| `buggy` | **label** | True if the commit was found to be bug-inducing |
| `fix` | to be checked | flag related to fixing commits; its exact meaning must be confirmed in the dataset readme before we use it as a feature |
| `year` | time | year of the commit |
| `author_date` | time | commit time as a Unix timestamp (seconds); used for ordering |
| `la`, `ld` | feature | lines added / lines deleted |
| `nf`, `nd`, `ns` | feature | files / directories / subsystems touched |
| `ent` | feature | entropy (spread) of the change |
| `ndev` | feature | number of distinct developers who touched the files |
| `age` | feature | average time since the files were last changed |
| `nuc` | feature | number of unique changes to the files |
| `aexp`, `arexp`, `asexp` | feature | author experience / recent experience / subsystem experience |

## 2. Label balance and data quality

**Goal:** see how many commits are bug-inducing, whether this differs across projects, and whether the data has missing values or duplicates.

The cell prints, in order: the label counts, the share of bug-inducing commits, a per-project table (`count` = number of commits, `mean` = share of bug-inducing commits), and two quality numbers (missing cells, duplicated commit ids).

In [7]:
print(df["buggy"].value_counts())
print(df["buggy"].mean())

print(df.groupby("project")["buggy"].agg(["count", "mean"]).round(2))

print(df.isna().sum().sum(), df["commit_id"].duplicated().sum())

buggy
False    78435
True     28239
Name: count, dtype: int64
0.26472242533325835
                         count  mean
project                             
apache/activemq           6126  0.23
apache/camel             22700  0.14
apache/cassandra          8159  0.38
apache/flink             11691  0.24
apache/groovy             8059  0.20
apache/hadoop            11964  0.00
apache/hadoop-hdfs        2907  0.76
apache/hadoop-mapreduce   1321  0.63
apache/hbase              8730  0.43
apache/hive               6842  0.62
apache/ignite            12036  0.20
apache/kafka              2384  0.47
apache/spark              1465  0.43
apache/zeppelin           1451  0.43
apache/zookeeper           839  0.41
0 0


**Observation**
- There are 78,435 clean and 28,239 bug-inducing commits (about 26.5%), exactly as in the paper.
- A model that always predicts "clean" would reach about 73.5% accuracy, so accuracy is not a useful main metric. We will use F1, MCC, PR-AUC and effort-aware metrics instead.
- The share of bug-inducing commits depends strongly on the project: from about 14% (Camel) to about 76% (Hadoop HDFS).
- `apache/hadoop` (11,964 commits) has almost no bug-inducing commits. The three Hadoop names (`hadoop`, `hadoop-hdfs`, `hadoop-mapreduce`) add up to 16,192 commits, which equals the HDFS and MapReduce rows of the paper together. We therefore treat them as one group in per-project analysis. The reason for the split in the file is not explained in the paper.
- There are no missing values and no duplicated commit ids (the last line prints `0 0`).

## 3. Do labels change over time?

**Goal:** check whether the share of bug-inducing commits is stable across years. This matters because we want to train on older commits and test on newer ones, and unstable labels would make that comparison unfair.

The table shows, for each year, the number of commits (`count`) and the share of bug-inducing commits (`mean`). `.to_string()` prints the whole table without hiding the middle rows.

In [8]:
print(df.groupby("year")["buggy"].agg(["count", "mean"]).round(2).to_string())

      count  mean
year             
2003    295  0.06
2004    517  0.07
2005    717  0.10
2006   1417  0.12
2007   2331  0.15
2008   3218  0.18
2009   5305  0.26
2010   4973  0.40
2011   7117  0.32
2012   6923  0.33
2013   7001  0.37
2014  12319  0.27
2015  13824  0.30
2016  10606  0.29
2017  10187  0.27
2018   9365  0.21
2019  10559  0.10


**Observation:** the share is not stable. It is about 6% in 2003, 27% in 2017, 21% in 2018 and only 10% in 2019.

Likely reasons (from the paper's construction process):
- Bug reports were collected only for 2010 to 2019, so early commits are under-labeled.
- A commit is labeled bug-inducing only after a later fix links back to it. Recent commits had less time to be linked, so some of them are probably labeled clean by mistake.

**Implication:** the newest years are a risky test set, because their labels are incomplete and the model would look worse than it really is. The exact cut points for train, validation and test are decided after reviewing the full table above.

## 4. Summary and decisions

- Keep all 106,674 rows; the data is complete and clean.
- Label: `buggy` (about 26.5% positive). Do not use accuracy as the main metric.
- Time: order commits by `author_date` and use a time-based split instead of a random split.
- Treat `hadoop`, `hadoop-hdfs` and `hadoop-mapreduce` as one group in per-project analysis.
- Open items: confirm what the `fix` column means (see the dataset readme) and decide the time cut points.

**Next:** `02_split_and_baseline.ipynb` - build the time-based split and train the first baseline (logistic regression).